<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Deteccion_de_Vehiculos_Detectron2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyyaml

In [2]:
!git clone 'https://github.com/facebookresearch/detectron2'

Cloning into 'detectron2'...
remote: Enumerating objects: 16057, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 16057 (delta 23), reused 7 (delta 7), pack-reused 16020 (from 3)
Receiving objects: 100% (16057/16057), 6.83 MiB | 22.85 MiB/s, done.
Resolving deltas: 100% (11379/11379), done.


In [3]:
import sys, os, distutils.core

dist = distutils.core.run_setup("./detectron2/setup.py")

In [4]:
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

Ignoring dataclasses: markers 'python_version < "3.7"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.8/155.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.4/268.4 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.4 MB/s eta 0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=9c14cf485427ca923db26377a0b67950a3845920ac736515026f516ce8729457
  Stored in directory: /root/.cache/pip/wheels/6b/1b/9a/9f4843148961a522f56dbeaa5543071ac2389d9a079ec724ab
Successfully built fvcore


In [5]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
torch:  2.11 ; cuda:  cu128
detectron2: 0.6


In [6]:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

In [7]:
# ==========================================
# CELDA 2: Descargar un video de prueba (Opcional)
# ==========================================
# Descargamos un video corto con tráfico y peatones
!wget -O video_entrada.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4

--2026-08-27 13:23:28--  https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/car-detection.mp4 [following]
--2026-08-27 13:23:29--  https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/car-detection.mp4
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2811553 (2.7M) [application/octet-stream]
Saving to: ‘video_entrada.mp4’

video_entrada.mp4   100%[===================>]   2.68M  --.-KB/s    in 0.009s  

2026-08-27 13:23:29 (312 MB/s) - ‘video_entrada.mp4’ saved [

In [7]:
!pip install supervision -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.6/376.6 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 15.3 MB/s eta 0:00:00


In [12]:
!pip install -q trackers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.8/220.8 kB 8.6 MB/s eta 0:00:00


In [18]:
import cv2
import torch
import numpy as np
import supervision as sv

from trackers import ByteTrackTracker

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor

from collections import deque

In [19]:
# 1. CONFIGURACIÓN DE DETECTRON2

cfg = get_cfg()
cfg.merge_from_file(
    model_zoo.get_config_file(
        "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
    )
)
# Umbral de confianza
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
# Pesos del modelo
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
)
# GPU / CPU
cfg.MODEL.DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Dispositivo utilizado:", cfg.MODEL.DEVICE)

predictor = DefaultPredictor(cfg)

Dispositivo utilizado: cuda
[08/27 15:18:38 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from https://dl.fbaipublicfiles.com/detectron2/COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl ...


In [21]:
# 2. BYTE TRACK
tracker = ByteTrackTracker()
print("ByteTrackTracker inicializado correctamente")

# 3. CARGAR VIDEO
video_path = "slow_traffic_small.mp4"
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise Exception(
        "No se pudo abrir el video."
    )

# 4. INFORMACIÓN DEL VIDEO
width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)
height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)
fps = cap.get(
    cv2.CAP_PROP_FPS
)
if fps <= 0:
    fps = 30.0
total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

print()
print("INFORMACIÓN DEL VIDEO")
print("Resolución:", width, "x", height)
print("FPS:", fps)
print("Frames:", total_frames)

ByteTrackTracker inicializado correctamente

INFORMACIÓN DEL VIDEO
Resolución: 640 x 360
FPS: 29.97002997002997
Frames: 914


In [ ]:
# 5. VIDEO DE SALIDA

output_path = (
    "video_personas_vehiculos_bytetrack.mp4"
)

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

if not out.isOpened():

    raise Exception(
        "No se pudo crear el video de salida."
    )


In [ ]:
# 6. CLASES COCO

# 0 = Persona
# 2 = Coche
# 3 = Motocicleta
# 5 = Autobús
# 7 = Camión

PERSONA = 0

VEHICULOS = {
    2: "Coche",
    3: "Moto",
    5: "Autobus",
    7: "Camion"
}

# Clases que vamos a detectar
CLASES_INTERES = [
    PERSONA,
    2,
    3,
    5,
    7
]

In [ ]:
# 7. PROCESAMIENTO

frame_count = 0

# ESTABILIZACIÓN DEL DASHBOARD

WINDOW_SIZE = 15

historial_personas = deque(
    maxlen=WINDOW_SIZE
)

historial_vehiculos = deque(
    maxlen=WINDOW_SIZE
)

personas_dashboard = 0
vehiculos_dashboard = 0

In [17]:
print("INICIANDO PROCESAMIENTO")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 7.1 DETECCIÓN

    outputs = predictor(frame)

    instances = (
        outputs["instances"]
        .to("cpu")
    )

    # 7.2 EXTRAER DATOS

    classes = (
        instances
        .pred_classes
        .numpy()
    )

    scores = (
        instances
        .scores
        .numpy()
    )

    boxes = (
        instances
        .pred_boxes
        .tensor
        .numpy()
    )


    # 7.3 FILTRAR SOLO PERSONAS Y VEHÍCULOS

    keep_mask = np.isin(
        classes,
        CLASES_INTERES
    )


    classes = classes[
        keep_mask
    ]

    scores = scores[
        keep_mask
    ]

    boxes = boxes[
        keep_mask
    ]


    # 7.4 CREAR DETECTIONS

    detections = sv.Detections(
        xyxy=boxes,
        confidence=scores,
        class_id=classes.astype(
            int
        )
    )


    # 7.5 BYTE TRACK

    tracked_detections = tracker.update(
        detections
    )


    # 7.6 FRAME RESULTADO

    processed_frame = frame.copy()


    # 7.7 DIBUJAR OBJETOS + ID

    if (
        tracked_detections.tracker_id
        is not None
    ):

        for i in range(
            len(tracked_detections)
        ):

            # COORDENADAS
            x1, y1, x2, y2 = (

                tracked_detections
                .xyxy[i]
                .astype(int)
            )

            # CLASE

            class_id = int(

                tracked_detections
                .class_id[i]
            )

            # ID

            tracker_id = int(

                tracked_detections
                .tracker_id[i]
            )

            # NOMBRE

            if class_id == PERSONA:
                nombre = "Persona"

            else:

                nombre = VEHICULOS.get(
                    class_id,
                    "Vehiculo"
                )

            # COLOR

            if class_id == PERSONA:

                color = (
                    255,
                    120,
                    0
                )

            else:

                color = (
                    0,
                    255,
                    0
                )

            # RECTÁNGULO
            cv2.rectangle(

                processed_frame,

                (x1, y1),

                (x2, y2),

                color,

                2
            )

            # ETIQUETA

            label = (

                f"{nombre} "
                f"#{tracker_id}"
            )

            # TEXTO
            (
                text_width,
                text_height
            ), baseline = (
                cv2.getTextSize(
                    label,
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45,
                    1
                )
            )

            # POSICIÓN ETIQUETA
            label_y1 = max(
                0,
                y1 -
                text_height -
                6
            )

            # FONDO ETIQUETA

            cv2.rectangle(
                processed_frame,
                (
                    x1,
                    label_y1
                ),

                (
                    x1 +
                    text_width +
                    4,
                    y1
                ),
                color,
                -1
            )

            # TEXTO ID

            cv2.putText(
                processed_frame,
                label,

                (
                    x1 + 2,
                    y1 - 3
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (0, 0, 0),
                1,
                cv2.LINE_AA
            )


    # 7.8 CONTADORES ACTUALES

    personas_actuales = 0
    vehiculos_actuales = 0

    if tracked_detections.tracker_id is not None:
        for class_id in (
            tracked_detections
            .class_id
        ):
            # Persona
            if class_id == PERSONA:
                personas_actuales += 1
            # Vehículos
            elif class_id in VEHICULOS:

                vehiculos_actuales += 1

    # ESTABILIZACIÓN TEMPORAL

    historial_personas.append(
        personas_actuales
    )
    historial_vehiculos.append(
    vehiculos_actuales
    )

    # Utilizamos la mediana
    personas_dashboard = int(
        np.median(
            list(historial_personas)
        )
    )
    vehiculos_dashboard = int(
        np.median(
            list(historial_vehiculos)
        )
    )

    # 7.9 DASHBOARD

    panel_width = 220
    panel_height = 75

    # Copia para transparencia
    overlay = (
        processed_frame.copy()
    )

    # PANEL NEGRO
    cv2.rectangle(
        overlay,
        (8, 8),
        (
            panel_width,
            panel_height
        ),
        (0, 0, 0),
        -1
    )

    # TRANSPARENCIA

    processed_frame = (
        cv2.addWeighted(

            overlay,

            0.60,

            processed_frame,

            0.40,

            0
        )
    )

    # PERSONAS

    cv2.putText(
        processed_frame,
        f"Personas: {personas_dashboard}",
        (18, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.58,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )

    # VEHÍCULOS

    cv2.putText(
        processed_frame,
        f"Vehiculos: {vehiculos_dashboard}",
        (18, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.58,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )

    # FRAME

    frame_count += 1
    cv2.putText(
        processed_frame,
        f"{frame_count}/{total_frames}",
        (
            width - 110,
            height - 15
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )

    # GUARDAR FRAME

    out.write(
        processed_frame
    )

    # PROGRESO

    if frame_count % 30 == 0:
        porcentaje = (
            frame_count /
            total_frames *
            100
        )

        print(
            f"{porcentaje:5.1f}% | "
            f"Personas: "
            f"{personas_actuales} | "
            f"Vehiculos: "
            f"{vehiculos_actuales}"

        )

# 8. FINALIZAR

cap.release()
out.release()
cv2.destroyAllWindows()

# 9. RESULTADO

print("PROCESAMIENTO FINALIZADO")
print("Video generado:")
print(output_path)


Dispositivo utilizado: cuda
[08/27 15:02:16 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from https://dl.fbaipublicfiles.com/detectron2/COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl ...
ByteTrackTracker inicializado correctamente

INFORMACIÓN DEL VIDEO
Resolución: 640 x 360
FPS: 29.97002997002997
Frames: 914

INICIANDO PROCESAMIENTO
  3.3% | Personas: 3 | Vehiculos: 8
  6.6% | Personas: 3 | Vehiculos: 8
  9.8% | Personas: 2 | Vehiculos: 8
 13.1% | Personas: 2 | Vehiculos: 8
 16.4% | Personas: 2 | Vehiculos: 8
 19.7% | Personas: 5 | Vehiculos: 9
 23.0% | Personas: 3 | Vehiculos: 9
 26.3% | Personas: 4 | Vehiculos: 9
 29.5% | Personas: 5 | Vehiculos: 9
 32.8% | Personas: 4 | Vehiculos: 10
 36.1% | Personas: 4 | Vehiculos: 11
 39.4% | Personas: 5 | Vehiculos: 8
 42.7% | Personas: 3 | Vehiculos: 8
 46.0% | Personas: 3 | Vehiculos: 9
 49.2% | Personas: 6 | Vehiculos: 10
 52.5% | Personas: 3 | Vehiculos: 10
 55.8% | Personas: 4 | Veh

In [8]:
import cv2
import torch
import numpy as np
import supervision as sv

# IMPORTACIONES DE DETECTRON2

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.data import MetadataCatalog

# 1. CONFIGURACIÓN DEL MODELO DETECTRON2

cfg = get_cfg()

cfg.merge_from_file(
    model_zoo.get_config_file(
        "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
    )
)

# Umbral de confianza
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

# Pesos del modelo
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
)

# GPU si está disponible
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo utilizado:", cfg.MODEL.DEVICE)

predictor = DefaultPredictor(cfg)
metadata = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])

# 2. CONFIGURAR BYTE TRACK

tracker = sv.ByteTrack()

print("ByteTrack inicializado correctamente")


# ============================================================
# 3. CARGAR VIDEO
# ============================================================

video_path = "slow_traffic_small.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise Exception("No se pudo abrir el video.")


# ============================================================
# 4. INFORMACIÓN DEL VIDEO
# ============================================================

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 30

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)


# ============================================================
# 5. CONFIGURAR VIDEO DE SALIDA
# ============================================================

output_path = "video_detectron2_bytetrack.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)


# ============================================================
# 6. CLASES COCO
# ============================================================

# COCO:
#
# 0 = Persona
# 1 = Bicicleta
# 2 = Coche
# 3 = Motocicleta
# 5 = Autobús
# 7 = Camión

PERSONA = 0

VEHICULOS = {
    1: "Bicicleta",
    2: "Coche",
    3: "Motocicleta",
    5: "Autobus",
    7: "Camion"
}

# Todas las clases que queremos rastrear
CLASES_INTERES = [PERSONA] + list(VEHICULOS.keys())


# ============================================================
# 7. VARIABLES DE CONTEO
# ============================================================

frame_count = 0

# IDs que hemos visto durante todo el video
personas_ids = set()
vehiculos_ids = set()

# IDs por categoría
ids_por_categoria = {
    "Persona": set(),
    "Bicicleta": set(),
    "Coche": set(),
    "Motocicleta": set(),
    "Autobus": set(),
    "Camion": set()
}


# ============================================================
# 8. PROCESAMIENTO FRAME POR FRAME
# ============================================================

print()
print("==============================================")
print("INICIANDO PROCESAMIENTO")
print("==============================================")
print(f"Frames totales: {total_frames}")
print(f"FPS: {fps:.2f}")
print(f"Resolución: {width}x{height}")
print()


while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break


    # ========================================================
    # 8.1 DETECCIÓN CON DETECTRON2
    # ========================================================

    outputs = predictor(frame)

    instances = outputs["instances"].to("cpu")


    # ========================================================
    # 8.2 OBTENER DATOS DE DETECTRON2
    # ========================================================

    classes = instances.pred_classes.numpy()

    scores = instances.scores.numpy()

    boxes = instances.pred_boxes.tensor.numpy()


    # ========================================================
    # 8.3 FILTRAR PERSONAS Y VEHÍCULOS
    # ========================================================

    keep_mask = np.isin(
        classes,
        CLASES_INTERES
    )

    classes = classes[keep_mask]

    scores = scores[keep_mask]

    boxes = boxes[keep_mask]


    # ========================================================
    # 8.4 CONVERTIR A FORMATO SUPERVISION
    # ========================================================

    detections = sv.Detections(
        xyxy=boxes,
        confidence=scores,
        class_id=classes.astype(int)
    )


    # ========================================================
    # 8.5 BYTE TRACK ASIGNA LOS IDs
    # ========================================================

    tracked_detections = tracker.update_with_detections(
        detections
    )


    # ========================================================
    # 8.6 OBTENER INFORMACIÓN DEL TRACKING
    # ========================================================

    if tracked_detections.tracker_id is not None:

        for i in range(
            len(tracked_detections)
        ):

            tracker_id = int(
                tracked_detections.tracker_id[i]
            )

            class_id = int(
                tracked_detections.class_id[i]
            )


            # ------------------------------------------------
            # PERSONA
            # ------------------------------------------------

            if class_id == PERSONA:

                personas_ids.add(
                    tracker_id
                )

                ids_por_categoria[
                    "Persona"
                ].add(tracker_id)


            # ------------------------------------------------
            # VEHÍCULOS
            # ------------------------------------------------

            elif class_id in VEHICULOS:

                vehiculos_ids.add(
                    tracker_id
                )

                nombre = VEHICULOS[
                    class_id
                ]

                ids_por_categoria[
                    nombre
                ].add(tracker_id)


    # ========================================================
    # 8.7 DIBUJAR DETECCIONES
    # ========================================================

    processed_frame = frame.copy()


    if tracked_detections.tracker_id is not None:

        for i in range(
            len(tracked_detections)
        ):

            # ----------------------------------------------
            # DATOS DEL OBJETO
            # ----------------------------------------------

            x1, y1, x2, y2 = (
                tracked_detections.xyxy[i]
                .astype(int)
            )

            class_id = int(
                tracked_detections.class_id[i]
            )

            tracker_id = int(
                tracked_detections.tracker_id[i]
            )

            confidence = float(
                tracked_detections.confidence[i]
            )


            # ----------------------------------------------
            # NOMBRE DE LA CLASE
            # ----------------------------------------------

            if class_id == PERSONA:

                nombre = "Persona"

            else:

                nombre = VEHICULOS.get(
                    class_id,
                    "Objeto"
                )


            # ----------------------------------------------
            # ETIQUETA
            # ----------------------------------------------

            label = (
                f"{nombre} #{tracker_id} "
                f"{confidence:.2f}"
            )


            # ----------------------------------------------
            # DIBUJAR RECTÁNGULO
            # ----------------------------------------------

            cv2.rectangle(
                processed_frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )


            # ----------------------------------------------
            # FONDO DE LA ETIQUETA
            # ----------------------------------------------

            (text_width, text_height), baseline = (
                cv2.getTextSize(
                    label,
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.55,
                    2
                )
            )


            cv2.rectangle(
                processed_frame,
                (x1, max(0, y1 - text_height - 10)),
                (
                    x1 + text_width + 5,
                    y1
                ),
                (0, 255, 0),
                -1
            )


            # ----------------------------------------------
            # TEXTO
            # ----------------------------------------------

            cv2.putText(
                processed_frame,
                label,
                (
                    x1 + 2,
                    y1 - 5
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (0, 0, 0),
                2,
                cv2.LINE_AA
            )


    # ========================================================
    # 8.8 CONTADORES ACTUALES
    # ========================================================

    personas_actuales = 0
    vehiculos_actuales = 0

    if tracked_detections.tracker_id is not None:

        for class_id in tracked_detections.class_id:

            if class_id == PERSONA:

                personas_actuales += 1

            elif class_id in VEHICULOS:

                vehiculos_actuales += 1


    # ========================================================
    # 8.9 PANEL DE INFORMACIÓN
    # ========================================================

    overlay = processed_frame.copy()

    panel_width = 410
    panel_height = 310

    cv2.rectangle(
        overlay,
        (10, 10),
        (
            panel_width,
            panel_height
        ),
        (0, 0, 0),
        -1
    )

    processed_frame = cv2.addWeighted(
        overlay,
        0.65,
        processed_frame,
        0.35,
        0
    )


    # ========================================================
    # TÍTULO
    # ========================================================

    cv2.putText(
        processed_frame,
        "DETECTRON2 + BYTETRACK",
        (25, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.75,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )


    # ========================================================
    # PERSONAS ACTUALES
    # ========================================================

    cv2.putText(
        processed_frame,
        f"Personas actuales: {personas_actuales}",
        (25, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )


    # ========================================================
    # PERSONAS ÚNICAS
    # ========================================================

    cv2.putText(
        processed_frame,
        f"Personas detectadas: {len(personas_ids)}",
        (25, 105),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )


    # ========================================================
    # VEHÍCULOS ACTUALES
    # ========================================================

    cv2.putText(
        processed_frame,
        f"Vehiculos actuales: {vehiculos_actuales}",
        (25, 140),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )


    # ========================================================
    # VEHÍCULOS ÚNICOS
    # ========================================================

    cv2.putText(
        processed_frame,
        f"Vehiculos detectados: {len(vehiculos_ids)}",
        (25, 170),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )


    # ========================================================
    # DESGLOSE DE VEHÍCULOS
    # ========================================================

    y = 205

    for nombre in [
        "Bicicleta",
        "Coche",
        "Motocicleta",
        "Autobus",
        "Camion"
    ]:

        cantidad = len(
            ids_por_categoria[nombre]
        )

        if cantidad > 0:

            cv2.putText(
                processed_frame,
                f"{nombre}: {cantidad}",
                (25, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                1,
                cv2.LINE_AA
            )

            y += 22


    # ========================================================
    # FRAME
    # ========================================================

    frame_count += 1

    cv2.putText(
        processed_frame,
        f"Frame: {frame_count}/{total_frames}",
        (
            width - 250,
            height - 20
        ),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )


    # ========================================================
    # GUARDAR FRAME
    # ========================================================

    out.write(
        processed_frame
    )


    # ========================================================
    # MOSTRAR PROGRESO
    # ========================================================

    if frame_count % 30 == 0:

        porcentaje = (
            frame_count /
            total_frames *
            100
        )

        print(
            f"Procesando: {porcentaje:.1f}% | "
            f"Personas actuales: {personas_actuales} | "
            f"Vehículos actuales: {vehiculos_actuales} | "
            f"Personas únicas: {len(personas_ids)} | "
            f"Vehículos únicos: {len(vehiculos_ids)}"
        )


# ============================================================
# 9. FINALIZAR
# ============================================================

cap.release()

out.release()

cv2.destroyAllWindows()


# ============================================================
# 10. RESULTADOS
# ============================================================

print()
print("==============================================")
print("PROCESAMIENTO FINALIZADO")
print("==============================================")

print(
    f"Personas únicas detectadas: "
    f"{len(personas_ids)}"
)

print(
    f"Vehículos únicos detectados: "
    f"{len(vehiculos_ids)}"
)

print()
print("Desglose:")

for nombre, ids in ids_por_categoria.items():

    print(
        f"  {nombre}: {len(ids)}"
    )

print()
print(
    "Video generado:",
    output_path
)



Dispositivo utilizado: cuda
[08/27 14:33:45 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from https://dl.fbaipublicfiles.com/detectron2/COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl ...


model_final_f10217.pkl: 178MB [00:03, 55.6MB/s]                           
/tmp/ipykernel_4861/1913117551.py:50: FutureWarning: The `ByteTrack` was deprecated since v0.28.0. It will be removed in v0.31.0.
  tracker = sv.ByteTrack()


ByteTrack inicializado correctamente

INICIANDO PROCESAMIENTO
Frames totales: 914
FPS: 29.97
Resolución: 640x360



/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0827 14:33:50.766000 4861 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


Procesando: 3.3% | Personas actuales: 3 | Vehículos actuales: 8 | Personas únicas: 4 | Vehículos únicos: 11
Procesando: 6.6% | Personas actuales: 3 | Vehículos actuales: 8 | Personas únicas: 4 | Vehículos únicos: 11
Procesando: 9.8% | Personas actuales: 2 | Vehículos actuales: 9 | Personas únicas: 4 | Vehículos únicos: 12
Procesando: 13.1% | Personas actuales: 2 | Vehículos actuales: 9 | Personas únicas: 6 | Vehículos únicos: 14
Procesando: 16.4% | Personas actuales: 2 | Vehículos actuales: 9 | Personas únicas: 9 | Vehículos únicos: 15
Procesando: 19.7% | Personas actuales: 5 | Vehículos actuales: 10 | Personas únicas: 12 | Vehículos únicos: 16
Procesando: 23.0% | Personas actuales: 3 | Vehículos actuales: 8 | Personas únicas: 13 | Vehículos únicos: 17
Procesando: 26.3% | Personas actuales: 4 | Vehículos actuales: 9 | Personas únicas: 13 | Vehículos únicos: 18
Procesando: 29.5% | Personas actuales: 5 | Vehículos actuales: 9 | Personas únicas: 15 | Vehículos únicos: 18
Procesando: 32.8%